# 02 — Classical baselines

Brute force, greedy top-K (Sharpe), Markowitz-then-round, and simulated
annealing — all on the same cardinality-constrained QUBO. Brute force is
the ground truth at the canonical instance (n=16, K=4, ~65k bitstrings).

Methodology: SA is QAOA's real classical analogue, so we report 10 seeds
(median + IQR) for it just like we will for QAOA.

> **Runtime:** ~10 s on Colab CPU.

In [1]:
# === Bootstrap (Colab + local) ===
import os, urllib.request as _u
exec((open('../scripts/bootstrap.py') if os.path.exists('../scripts/bootstrap.py') else _u.urlopen('https://raw.githubusercontent.com/egil10/fys5419/main/project2/code/scripts/bootstrap.py')).read())

# === Project imports ===
import numpy as np
import pandas as pd

from scripts.data      import load_universe
from scripts.portfolio import PortfolioProblem, DEFAULTS
from scripts.classical import (
    brute_force, greedy_top_k, markowitz_round, simulated_annealing,
)
from scripts.metrics   import sharpe

> /usr/bin/python3 -m pip install -q numpy pandas scipy matplotlib yfinance pyarrow
[colab.setup] env=Colab
[colab.setup] cwd  = /content/fys5419/project2/code/notebooks
[colab.setup] path = /content/fys5419/project2/code  (added to sys.path)


### Canonical instance — `n=16, K=4`

All 16 equities (Mag7 + quantum + quantum_big + anti), λ and A from
`DEFAULTS`, K = `DEFAULTS["K_AT_16"] = 4`. Same problem the QAOA notebooks
(03, 04, 06) use.

In [2]:
# === Canonical instance: n=16, K=4 ===
r  = load_universe()
pf = PortfolioProblem(
    r.mu, r.Sigma,
    lam=DEFAULTS['lam'], A=DEFAULTS['A'],
    K=DEFAULTS['K_AT_16'],
    tickers=r.tickers,
)
print(pf)
print(f'universe ({r.n} assets):', r.tickers)

⟳ cache stale (cache starts 2023-01-03 after requested 2023-01-01); re-fetching
✓ 751 1d obs for ['AAPL', 'MSFT', 'GOOGL', 'AMZN', 'IONQ', 'RGTI', 'QBTS', 'QUBT', 'IBM', 'HON', 'ACN', 'NVDA', 'XOM', 'DAL', 'NEM', 'JPM'] (2023-01-03 → 2025-12-30)
✓ cached → data/universe_16.parquet
✓ sample → data/universe_16_sample.csv
PortfolioProblem(n=16, K=4, lam=2.0, A=0.5)
universe (16 assets): ('AAPL', 'MSFT', 'GOOGL', 'AMZN', 'IONQ', 'RGTI', 'QBTS', 'QUBT', 'IBM', 'HON', 'ACN', 'NVDA', 'XOM', 'DAL', 'NEM', 'JPM')


### Deterministic baselines (single run)

In [ ]:
from scripts.mip import mip_exact

det = {
    'brute_force':     brute_force(pf),
    'greedy_sharpe':   greedy_top_k(pf, score='sharpe'),
    'markowitz_round': markowitz_round(pf),
    # MIP via PuLP/CBC — the industry baseline for cardinality-constrained
    # mean-variance portfolio selection. Same QUBO, exact integer feasibility.
    'mip_cbc':         mip_exact(pf, time_limit=60.0),
}
rows = [{
    'solver':    sr.name,
    'bitstring': sr.bitstring,
    'tickers':   '+'.join(sr.tickers(pf)),
    'cost':      sr.cost,
    'feasible':  sr.feasible,
    'sharpe':    sharpe(sr.x, pf.mu, pf.Sigma),
    'runtime_s': sr.runtime,
} for sr in det.values()]
pd.DataFrame(rows).sort_values('cost')

### Simulated annealing — 10 seeds, report median + IQR

In [ ]:
N_SEEDS = 10
# n_sweeps = 1000*n matches notebooks 05_scaling and 06_risk so the
# runtime column in tab:classical-baselines is comparable across notebooks.
sa_runs = [simulated_annealing(pf, n_sweeps=1000*pf.n, seed=s) for s in range(N_SEEDS)]
sa_costs = np.array([sr.cost for sr in sa_runs])
best_sa  = min(sa_runs, key=lambda sr: sr.cost)

q25, q50, q75 = np.percentile(sa_costs, [25, 50, 75])
print(f'SA over {N_SEEDS} seeds (n_sweeps = 1000*n = {1000*pf.n}):')
print(f'  median cost: {q50:.6f}  (IQR {q25:.6f} -> {q75:.6f})')
print(f'  best  cost:  {best_sa.cost:.6f}  ({best_sa.bitstring})')
print(f'  brute force: {det["brute_force"].cost:.6f}')